# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [1]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load hotel booking data from GitHub
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(url, na_values=["NULL"])

print("Dataset shape:", df.shape)
display(df.head())

# Remove columns that could cause data leakage
df = df.drop(columns=[
    "reservation_status",
    "reservation_status_date",
    "assigned_room_type"
])

# Define features and target
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

# Split BEFORE preprocessing (70% training, 30% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Identify numerical and categorical columns
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Handle missing numerical values and scale features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Handle missing categorical values and encode categories
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant", fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Create preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training set:", X_train_processed.shape)
print("Test set:", X_test_processed.shape)
print("Missing training values:",
      pd.isna(X_train_processed).sum())
print("Missing test values:",
      pd.isna(X_test_processed).sum())


Dataset shape: (9404, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


Training set: (6582, 130)
Test set: (2822, 130)
Missing training values: 0
Missing test values: 0


### ✍️ Your Response: 🔧
1. The dataset contains 9,404 rows and 32 columns.

2. The dataset includes both numerical and categorical features. Numerical features include lead time, number of adults, and average daily rate. Categorical features include hotel type, meal type, and market segment.

3. I removed columns that could cause data leakage, handled missing values using imputation, and converted categorical variables into numerical values using one-hot encoding. I also standardized numerical features and split the data into 70% training and 30% testing before preprocessing. After preprocessing, there were 130 features and no missing values.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [2]:

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Create and train the Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train_processed, y_train)

# Predict cancellations using the test data
y_pred_nb = nb_model.predict(X_test_processed)

# Evaluate the model
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))


Naive Bayes Accuracy: 0.4922041105598866

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.32      0.49      2121
           1       0.33      1.00      0.49       701

    accuracy                           0.49      2822
   macro avg       0.66      0.66      0.49      2822
weighted avg       0.83      0.49      0.49      2822


Confusion Matrix:
[[ 689 1432]
 [   1  700]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The Naïve Bayes model performed poorly overall, with an accuracy of 49.22%. It correctly identified almost all cancellations, with a recall of 1.00, but incorrectly predicted many bookings would be canceled. The F1-score is a better metric than accuracy because it balances precision and recall, which is important when evaluating cancellation predictions.

2. This model could be useful for identifying bookings that might be at risk of cancellation and sending early alerts to hotel management. However, because it produces so many false positives, it could lead to unnecessary staffing or inventory adjustments. The hotel would need a more accurate model before relying on it for major operational decisions.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [3]:

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Data was already scaled in Section 1 using StandardScaler
# and fitted only on the training data

# Create and train SVM model with linear kernel
svm_model = SVC(kernel="linear", random_state=42)
svm_model.fit(X_train_processed, y_train)

# Predict cancellations on the test set
y_pred_svm = svm_model.predict(X_test_processed)

# Evaluate model performance
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))


SVM Accuracy: 0.8979447200566973

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2121
           1       0.83      0.74      0.78       701

    accuracy                           0.90      2822
   macro avg       0.87      0.84      0.86      2822
weighted avg       0.90      0.90      0.90      2822


Confusion Matrix:
[[2016  105]
 [ 183  518]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The SVM model performed very well, with an accuracy of 89.79%. It correctly predicted 518 out of 701 cancellations and had an F1-score of 0.78 for canceled bookings. The F1-score is a good metric because it balances precision and recall, helping us understand how reliably the model predicts cancellations.

2. SVM could be useful when hotels need more accurate cancellation predictions to make staffing, inventory, and revenue decisions. It could help identify high-risk bookings and allow management to plan ahead. Compared to Naïve Bayes, SVM produces fewer false positives, making its predictions more useful for business decisions.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [4]:

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create a validation set from the training data (80/20)
X_nn_train, X_nn_val, y_nn_train, y_nn_val = train_test_split(
    X_train_processed,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

# Build a neural network with two hidden layers
nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42
)

# Train the model using training data only
nn_model.fit(X_nn_train, y_nn_train)

# Evaluate on the validation set
y_val_pred = nn_model.predict(X_nn_val)
print("Validation Accuracy:", accuracy_score(y_nn_val, y_val_pred))

# Predict on the test set
y_pred_nn = nn_model.predict(X_test_processed)

# Evaluate the model
print("\nNeural Network Test Accuracy:",
      accuracy_score(y_test, y_pred_nn))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nn))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nn))


Validation Accuracy: 0.9187547456340167

Neural Network Test Accuracy: 0.9089298369950389

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94      2121
           1       0.85      0.77      0.81       701

    accuracy                           0.91      2822
   macro avg       0.89      0.86      0.87      2822
weighted avg       0.91      0.91      0.91      2822


Confusion Matrix:
[[2022   99]
 [ 158  543]]


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The Neural Network performed the best out of all three models, with an accuracy of 90.89%, compared to 89.79% for SVM and 49.22% for Naïve Bayes. It also had the highest F1-score for predicting cancellations at 0.81. The model correctly identified 543 out of 701 cancellations, making it the most accurate and balanced model overall.

2. The business might be comfortable using a black box model because of its high accuracy, but there could be concerns about not understanding exactly how it makes predictions. This could make it harder for management to trust the results. I think the hotel could use the Neural Network as long as its performance is regularly monitored and the predictions are used to support decisions rather than make them automatically.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [5]:

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Compare the three models
results = pd.DataFrame({
    "Model": ["Naive Bayes", "SVM", "Neural Network"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_nn)
    ],
    "Precision": [
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_svm),
        precision_score(y_test, y_pred_nn)
    ],
    "Recall": [
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_svm),
        recall_score(y_test, y_pred_nn)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_svm),
        f1_score(y_test, y_pred_nn)
    ]
})

display(results.round(3))

# Identify best model
best_model = results.loc[results["Accuracy"].idxmax(), "Model"]
print("\nBest performing model:", best_model)


,Model,Accuracy,Precision,Recall,F1-Score
0,Naive Bayes,0.492,0.328,0.999,0.494
1,SVM,0.898,0.831,0.739,0.782
2,Neural Network,0.909,0.846,0.775,0.809



Best performing model: Neural Network


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. The Neural Network had the best overall accuracy at 90.89%, followed by SVM at 89.79% and Naïve Bayes at 49.22%. Naïve Bayes is the simplest and generally fastest model to train, while SVM and Neural Networks are more complex. Naïve Bayes is easier to understand, while the Neural Network is the hardest to interpret. Overall, the Neural Network performed best in terms of prediction accuracy.

2. I would recommend the Neural Network because it achieved the highest accuracy and F1-score and correctly identified more cancellations than SVM. This could help the hotel improve staffing, inventory management, and revenue planning. However, I would test it further before deploying it because it is harder to interpret and still misses some cancellations.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend using the Neural Network model because it achieved the highest accuracy at 90.89% and performed best at predicting cancellations overall. This could help hotels better manage staffing, room availability, and revenue by identifying bookings at risk of cancellation. However, the model is difficult to interpret and still makes some incorrect predictions. I would recommend testing it further before full implementation. In the future, adding more customer booking history, seasonal trends, and cancellation reasons could help improve accuracy.

2. This assignment relates to my customized learning outcome of using business analytics tools to analyze customer data, identify patterns, and make better marketing decisions. By comparing three classification models, I learned how businesses can use historical data to predict customer behavior. These predictions can help hotels make better decisions about staffing, pricing, and customer retention. It also showed me how data analysis can support business growth and improve operations.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"